In [11]:
import numpy as np
from sklearn.metrics import pairwise_distances, silhouette_score
import ollama
from sklearn.metrics.pairwise import cosine_similarity
import re

# -----------------------------
# Load datasets
# -----------------------------
with open("C:\\Users\\sidha\\Desktop\\Simplifying Ideas Differently\\thought space simulator\\Thought-Space-Simulator\\epistemoverse_tss\\texts\\Free_Will_ARISTOTLE.txt", "r", encoding="utf-8") as f:
    uncleaned_entries = f.read().split("\n\n")

with open("C:\\Users\\sidha\\Desktop\\Simplifying Ideas Differently\\thought space simulator\\Thought-Space-Simulator\\epistemoverse_tss\\texts\\Free_Will_ARISTOTLE_cleaned.txt", "r", encoding="utf-8") as f:
    cleaned_entries = f.read().split("\n\n")


In [10]:
# ---------------------------
# 1. Parse file into list of texts
# ---------------------------
def parse_text_blocks(path):
    """
    Parse a file of the form:
    Index: xxxx
    Keys: ...
    Text: ...
    Question: ...
    
    Returns list of text passages (Text + Question).
    """
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    # Split into blocks starting with "Index:"
    blocks = re.split(r"Index:\s*\d+", content)
    passages = []
    for block in blocks:
        if not block.strip():
            continue
        # Extract Text and Question
        text_match = re.search(r"Text:\s*(.*?)(?=Question:)", block, re.S)
        question_match = re.search(r"Question:\s*(.*)", block, re.S)
        if text_match:
            text = text_match.group(1).strip()
            question = question_match.group(1).strip() if question_match else ""
            passages.append(text + " " + question)
    return passages

# ---------------------------
# 2. Get embeddings with Ollama
# ---------------------------
def get_embeddings(texts, model="nomic-embed-text"):
    embeddings = []
    for t in texts:
        response = ollama.embeddings(model=model, prompt=t)
        embeddings.append(response["embedding"])
    return np.array(embeddings)

# ---------------------------
# 3. Scoring function
# ---------------------------
def compute_scores(embeddings, awareness_anchor, integration_anchor, alpha=0.5, beta=0.5):
    embeddings = np.array(embeddings)
    awareness = np.mean(cosine_similarity(embeddings, awareness_anchor.reshape(1, -1)))
    integration = np.mean(cosine_similarity(embeddings, integration_anchor.reshape(1, -1)))
    overall = alpha * awareness + beta * integration
    return {
        "awareness": float(awareness),
        "integration": float(integration),
        "overall": float(overall)
    }


In [14]:

# ---------------------------
# 4. Load data + anchors
# ---------------------------
aristotle_texts = parse_text_blocks("C:\\Users\\sidha\\Desktop\\Simplifying Ideas Differently\\thought space simulator\\Thought-Space-Simulator\\epistemoverse_tss\\texts\\Free_Will_ARISTOTLE.txt")
aristotle_texts_cleaned = parse_text_blocks("C:\\Users\\sidha\\Desktop\\Simplifying Ideas Differently\\thought space simulator\\Thought-Space-Simulator\\epistemoverse_tss\\texts\\Free_Will_ARISTOTLE_cleaned.txt")

awareness_anchor = np.load("C:\\Users\\sidha\\Desktop\\Simplifying Ideas Differently\\thought space simulator\\Thought-Space-Simulator\\epistemoverse_tss\\anchors\\awareness.npy")
integration_anchor = np.load("C:\\Users\\sidha\\Desktop\\Simplifying Ideas Differently\\thought space simulator\\Thought-Space-Simulator\\epistemoverse_tss\\anchors\\integration.npy")

# ---------------------------
# 5. Generate embeddings
# ---------------------------
aristotle_embeddings = get_embeddings(aristotle_texts)
aristotle_embeddings_cleaned = get_embeddings(aristotle_texts_cleaned)

# ---------------------------
# 6. Compute scores
# ---------------------------
scores_aristotle = compute_scores(aristotle_embeddings, awareness_anchor, integration_anchor)
scores_aristotle_cleaned = compute_scores(aristotle_embeddings_cleaned, awareness_anchor, integration_anchor)
# scores_combined = compute_scores(
#     np.vstack([aristotle_embeddings, aristotle_embeddings_cleaned]),
#     awareness_anchor,
#     integration_anchor
# )

print("Aristotle only:", scores_aristotle)
print("Aristotle Cleaned:", scores_aristotle_cleaned)


Aristotle only: {'awareness': 0.4430455322424402, 'integration': 0.47304952228705627, 'overall': 0.45804752726474823}
Aristotle Cleaned: {'awareness': 0.43542435428498194, 'integration': 0.409970085343337, 'overall': 0.4226972198141595}
